# **<span style="color:red"> -Scrapping Data From Naukri.com Website </span>**

<br>
<br>
<br>

### **Code**

In [1]:
import os
import time
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import google.generativeai as genai
from dotenv import load_dotenv  # Import dotenv to load environment variables

# -------------------------------
# 1. Configuration and Setup
# -------------------------------

# Load environment variables from the .env file
load_dotenv()

# Get the API key from the environment variable
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Check if API key is loaded
if not GEMINI_API_KEY:
    raise ValueError("Gemini API key not found. Please set it in the .env file.")

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# Configure Gemini AI
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(model_name="gemini-1.5-flash")


# Set up the Selenium WebDriver using Service and ChromeDriverManager
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# Initialize empty lists to store scraped data
CompanyName = []
JobTitle = []
Experience = []
Skills = []
Links = []

# -------------------------------
# 2. Web Scraping Function
# -------------------------------

def scrape_job_listings(page_number, job_type, url_pattern):
    job_url = url_pattern.format(page_number=page_number)
    
    logger.info(f"Fetching URL: {job_url}")
    
    # Open the webpage
    driver.get(job_url)
    
    # Wait for the page to load
    time.sleep(5)
    
    # Get the page source
    page_content = driver.page_source
    
    # Parse the HTML with BeautifulSoup
    soup = BeautifulSoup(page_content, "html.parser")
    
    # Find all job listing boxes on the page
    boxes = soup.find_all('div', class_="cust-job-tuple layout-wrapper lay-2 sjw__tuple")
    
    logger.info(f"Found {len(boxes)} {job_type} job listings on page {page_number}")
    
    # Extract data from each job listing
    for box in boxes:
        # Extract Company Name
        company = box.find('a', class_='comp-name')
        company_name = company.text.strip() if company else 'N/A'
        CompanyName.append(company_name)
        
        # Extract Job Title
        job = box.find('a', class_='title')
        job_title = job.text.strip() if job else 'N/A'
        JobTitle.append(job_title)
        
        # Extract Experience
        experience = box.find('span', class_='expwdth')
        experience_text = experience.text.strip() if experience else 'N/A'
        Experience.append(experience_text)
        
        # Extract Skills
        skills_list = box.find('ul', class_='tags-gt')
        if skills_list:
            skills = [skill.text.strip() for skill in skills_list.find_all('li')]
            skills_text = ', '.join(skills)
            Skills.append(skills_text)
        else:
            Skills.append('N/A')
        
        # Extract Links
        link = box.find('a', class_='title')
        link_url = link['href'] if link else 'N/A'
        Links.append(link_url)


# -------------------------------
# 3. Main Execution
# -------------------------------

def main():
    job_types = {
        "Data Science": "https://www.naukri.com/data-scientist-data-science-jobs-{page_number}?k=data%20scientist%2C%20data%20science&nignbevent_src=jobsearchDeskGNB",
        "Software Engineering": "https://www.naukri.com/software-engineering-jobs-{page_number}?k=software%20engineering&nignbevent_src=jobsearchDeskGNB",
        "Software Testing": "https://www.naukri.com/software-testing-jobs-{page_number}?k=software%20testing&nignbevent_src=jobsearchDeskGNB",
        "Cloud Engineering": "https://www.naukri.com/cloud-engineering-jobs-{page_number}?k=cloud%20engineering&nignbevent_src=jobsearchDeskGNB",
        "Machine Learning Engineering": "https://www.naukri.com/machine-learning-engineer-jobs-{page_number}?k=machine%20learning%20engineer&nignbevent_src=jobsearchDeskGNB",
        "AI Engineering": "https://www.naukri.com/ai-engineer-jobs-{page_number}?k=ai%20engineer&nignbevent_src=jobsearchDeskGNB",
        "Java Developers": "https://www.naukri.com/java-developer-jobs-{page_number}?k=java+developer&nignbevent_src=jobsearchDeskGNB"
    }
    
    # Define the range of pages you want to scrape manually
    pages_to_scrape = range(1, 21)
    
    # Step 1: Scrape job listings for each job type and page
    for job_type, url_pattern in job_types.items():
        logger.info(f"Starting to scrape {job_type} jobs.")
        for page in pages_to_scrape:
            scrape_job_listings(page, job_type, url_pattern)
    
    # Close the driver after scraping all pages
    driver.quit()
    logger.info("Completed web scraping and closed the browser.")
    
    # Step 2: Create a DataFrame from the scraped data
    job_data = {
        'CompanyName': CompanyName,
        'JobRole': JobTitle,
        'Experience': Experience,
        'Skills': Skills,
        'Links': Links
    }
    
    df = pd.DataFrame(job_data)
    logger.info("Created initial DataFrame.")
    print("Initial DataFrame:")
    print(df.head())  # Display the top rows of the DataFrame for reference
    
    # Step 3: Save the enriched DataFrame to a CSV file
    output_file = 'job_descriptions.csv'
    df.to_csv(output_file, index=False)
    logger.info(f"DataFrame saved to '{output_file}'.")

if __name__ == "__main__":
    main()

## 2787 Rows of Data Collected in 18 mins.

c:\Users\janum\Documents\1. Data Science\- College Project -\Main\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:WDM:====== WebDriver manager ======
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:There is no [win64] chromedriver "142.0.7444.61" for browser google-chrome "142.0.7444" in cache
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:WebDriver version 142.0.7444.61 selected
INFO:WDM:Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/142.0.7444.61/win32/chromedriver-win32.zip
INFO:WDM:About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/142.0.7444.61/win32/chromedriver-win32.zip
INFO:WDM:Driver downloading response is 200
INFO:WDM:Get L

Initial DataFrame:
    CompanyName                                            JobRole Experience  \
0     Capgemini                       Risk & Finance Data Engineer    4-9 Yrs   
1  WNS Holdings  Data Scientist - Marketing and Campaign Analytics    1-2 Yrs   
2       Nagarro                   Staff Engineer, Machine Learning    1-5 Yrs   
3     Accenture  S&C Global Network - AI - Retail - Consultant ...    4-9 Yrs   
4           IBM             Data Scientist-Artificial Intelligence    3-7 Yrs   

                                              Skills  \
0  data science, ml, python, machine learning, da...   
1  Python, SQL, Power Bi, Azure Cloud, Alteryx, M...   
2  technical writing, python, data analytics, dat...   
3  python, natural language processing, machine l...   
4  algorithms, python, data analytics, tableau, m...   

                                               Links  
0  https://www.naukri.com/job-listings-risk-finan...  
1  https://www.naukri.com/job-listings-data-sci